In [ ]:
a = 1

In [ ]:
a = 1

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")

default_plot_setting()

In [ ]:
df, df_part = read_comix_uk_contact_raw();

println("# contacts: ", nrow(df))
println("# participants (rows): ", nrow(df_part))

In [ ]:
df_chunk = create_week_df(df_part)
df_part = add_date_chunks(@subset(df_part,
    Date(2021, 7, 1) .<= :date .<= Date(2021, 12, 31)))

df = innerjoin(df, @select(df_part, :part_id_d, :date,
    :chunk_number, :chunk_start, :chunk_end, :mid_date),
    on = [:part_id_d, :date])

println("# weeks: ", nrow(df_chunk))
first(df_chunk, 5)

In [ ]:
is_na(v) = ismissing(v) || (v isa AbstractString && v == "NA")

n_total       = nrow(df)
n_miss_phys   = count(is_na, df[:, :phys_contact])
n_miss_dur    = count(is_na, df[:, :duration_multi])
n_home        = count(==("true"),  df[:, :cnt_home])
n_nonhome     = count(==("false"), df[:, :cnt_home])

println("contacts total           : ", n_total)
println("  missing phys_contact   : ", n_miss_phys)
println("  missing duration_multi : ", n_miss_dur)
println("  cnt_home == true       : ", n_home)
println("  cnt_home == false      : ", n_nonhome)

# Visualisation of proportions

In [ ]:
# Aggregated empirical proportions vs degree, three duration variants + phys.
# Helpers live in `vis_utils.jl` (included by `cell-setup`).
#   1. NA imputed as <5 min  (matches `prepare_dm_inputs` default).
#   2. NA kept as its own 6th category.
#   3. NA excluded from proportion (numerator AND denominator), but degree
#      (x-axis) still counts NA-duration contacts.
p_dur_home        = _plot_props_panel(df, "home",     :duration_multi, 5; title_suffix = " (NA→<5min)")
p_dur_non         = _plot_props_panel(df, "non-home", :duration_multi, 5; title_suffix = " (NA→<5min)")
p_dur_na_home     = _plot_props_panel_dur_na(df, "home")
p_dur_na_non      = _plot_props_panel_dur_na(df, "non-home")
p_dur_dropna_home = _plot_props_panel_dur_dropna(df, "home")
p_dur_dropna_non  = _plot_props_panel_dur_dropna(df, "non-home")
p_phys_home       = _plot_props_panel(df, "home",     :phys_contact,   2)
p_phys_non        = _plot_props_panel(df, "non-home", :phys_contact,   2)

fig_props_agg = plot(p_dur_home, p_dur_non,
     p_dur_na_home, p_dur_na_non,
     p_dur_dropna_home, p_dur_dropna_non,
     p_phys_home, p_phys_non;
    layout = (4, 2), size = (1100, 1400))

out_dir_fig = "../res/2j_proportion_duration_physical"
isdir(out_dir_fig) || mkpath(out_dir_fig)
savefig(fig_props_agg, joinpath(out_dir_fig, "props_aggregated.png"))

fig_props_agg

In [ ]:
# Raw per-cell proportions vs degree (no degree-aggregation).
# Helpers live in `vis_utils.jl` (included by `cell-setup`).
# Marker area ∝ log10(# cells stacked at that exact (degree, proportion) point).

# Duration: 5 categories × 2 settings (NA imputed as <5min).
plts_dur_raw = Plots.Plot[]
for k in 1:5
    push!(plts_dur_raw, _plot_raw_panel(df, "home",     :duration_multi, 5, k))
    push!(plts_dur_raw, _plot_raw_panel(df, "non-home", :duration_multi, 5, k))
end
fig_dur_raw = plot(plts_dur_raw...; layout = (5, 2), size = (1100, 1500),
    plot_title = "Raw per-cell duration proportions vs degree (NA→<5min)")

# Duration: 6 categories × 2 settings (NA as its own category).
plts_dur_raw_na = Plots.Plot[]
for k in 1:6
    push!(plts_dur_raw_na, _plot_raw_panel_dur_na(df, "home",     k))
    push!(plts_dur_raw_na, _plot_raw_panel_dur_na(df, "non-home", k))
end
fig_dur_raw_na = plot(plts_dur_raw_na...; layout = (6, 2), size = (1100, 1800),
    plot_title = "Raw per-cell duration proportions vs degree (NA as category)")

# Physical: 2 categories × 2 settings.
plts_phys_raw = Plots.Plot[]
for k in 1:2
    push!(plts_phys_raw, _plot_raw_panel(df, "home",     :phys_contact, 2, k))
    push!(plts_phys_raw, _plot_raw_panel(df, "non-home", :phys_contact, 2, k))
end
fig_phys_raw = plot(plts_phys_raw...; layout = (2, 2), size = (1100, 600),
    plot_title = "Raw per-cell physical-contact proportions vs degree")

out_dir_fig = "../res/2j_proportion_duration_physical"
isdir(out_dir_fig) || mkpath(out_dir_fig)
savefig(fig_dur_raw,    joinpath(out_dir_fig, "props_raw_duration.png"))
savefig(fig_dur_raw_na, joinpath(out_dir_fig, "props_raw_duration_na.png"))
savefig(fig_phys_raw,   joinpath(out_dir_fig, "props_raw_physical.png"))

display(fig_dur_raw)
display(fig_dur_raw_na)
display(fig_phys_raw)

# Dirichlet-multinomial regression: contact duration & physical contact proportions

Analysis for 2021-07 to 2021-12, separately for home vs non-home contacts.

- Outcome 1 — `duration_multi` (K=5): <5min, 5–15min, 15min–1hr, 1–4hr, 4+hr.
- Outcome 2 — `phys_contact` (K=2): physical, non-physical.
- Predictor: `log(degree)` per (participant, diary day, setting).
- Models: `model_dm_logdeg` (full) and `model_dm_logdeg_constprec` (constant precision); compared via WAIC.

In [ ]:
# Primary policy: impute missing duration as <5 min, drop missing phys_contact.
panels = Dict{Symbol, NamedTuple}()
for (panel, setting, outcome, K) in [
        (:dur_home,     "home",     :duration_multi, 5),
        (:dur_nonhome,  "non-home", :duration_multi, 5),
        (:phys_home,    "home",     :phys_contact,   2),
        (:phys_nonhome, "non-home", :phys_contact,   2),
    ]
    panels[panel] = prepare_dm_inputs(df; setting = setting, outcome = outcome, K = K)
    inp = panels[panel]
    println(panel, " -> N=", size(inp.X, 1), ", K=", K,
        ", n range=", extrema(inp.n))
end

In [ ]:
# Fitting config — bump n_sample for the production run.
n_sample = 500
K_for_panel = Dict(:dur_home => 5, :dur_nonhome => 5,
                   :phys_home => 2, :phys_nonhome => 2)

In [ ]:
# Fit the full degree-in-precision model for each panel.
chains_full = Dict{Symbol, Chains}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    println("Fitting full model for ", panel, " (N=", size(inp.X, 1), ", K=", K, ")")
    m = model_dm_logdeg(inp.X, inp.Y, inp.n; K = K)
    chains_full[panel] = fit_model_with_forward_mode(m, n_sample; progress = false)
end

In [ ]:
# Fit the constant-precision restriction (WAIC null for the degree term in the precision).
chains_constprec = Dict{Symbol, Chains}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    println("Fitting constprec model for ", panel)
    m = model_dm_logdeg_constprec(inp.X, inp.Y, inp.n; K = K)
    chains_constprec[panel] = fit_model_with_forward_mode(m, n_sample; progress = false)
end

In [ ]:
# Convergence diagnostics across all 8 fits.
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    println("=== ", panel, " — full ===")
    summarize_dm_fit(chains_full[panel])
    println("=== ", panel, " — constprec ===")
    summarize_dm_fit(chains_constprec[panel])
end

In [ ]:
# WAIC comparison: ΔWAIC = WAIC_constprec - WAIC_full (positive → full preferred).
waic_rows = NamedTuple[]
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    ll_full      = dm_log_lik_matrix(chains_full[panel],      inp.X, inp.Y, inp.n; K = K, mode = :full)
    ll_constprec = dm_log_lik_matrix(chains_constprec[panel], inp.X, inp.Y, inp.n; K = K, mode = :constprec)
    w_full      = calc_waic_from_loglik(ll_full)
    w_constprec = calc_waic_from_loglik(ll_constprec)
    Δwaic   = w_constprec.waic - w_full.waic
    se_diff = calc_waic_diff_se(w_full.elpd_i, w_constprec.elpd_i)
    push!(waic_rows, (panel = panel,
        waic_full = w_full.waic, waic_constprec = w_constprec.waic,
        Δwaic = Δwaic, se_diff = se_diff,
        prefer = Δwaic > 2 * se_diff ? "full" :
                 Δwaic < -2 * se_diff ? "constprec" : "tie"))
end
df_waic = DataFrame(waic_rows)
df_waic

In [ ]:
# Posterior predictive helpers: μ over a degree grid for the WAIC-preferred model per panel.
function pp_mu_curves(chn::Chains, K::Int, n_grid::AbstractVector{<:Integer}, mode::Symbol)
    df = DataFrame(chn)
    Xg = hcat(ones(length(n_grid)), log.(n_grid))
    β_cols = [Symbol("β[$i, $j]") for i in 1:2, j in 1:(K - 1)]
    S = nrow(df)
    μ_draws = Array{Float64}(undef, S, length(n_grid), K)
    for s in 1:S
        β = [df[s, β_cols[i, j]] for i in 1:2, j in 1:(K - 1)]
        μ_draws[s, :, :] = calc_dm_proportions(Xg, β)
    end
    μ_mean  = dropdims(mean(μ_draws, dims = 1), dims = 1)
    μ_lower = mapslices(v -> quantile(v, 0.025), μ_draws; dims = 1) |> x -> dropdims(x, dims = 1)
    μ_upper = mapslices(v -> quantile(v, 0.975), μ_draws; dims = 1) |> x -> dropdims(x, dims = 1)
    @assert all(abs.(sum(μ_mean, dims = 2) .- 1) .< 1e-8)
    return (μ_mean = μ_mean, μ_lower = μ_lower, μ_upper = μ_upper)
end

function empirical_proportions_by_n(Y::Matrix{Int}, n::Vector{Int}, K::Int)
    df_e = DataFrame(n = n)
    for k in 1:K; df_e[!, Symbol("y$k")] = Y[:, k]; end
    g = combine(groupby(df_e, :n)) do sub
        total = sum(sub[:, :n])
        (; (Symbol("p$k") => sum(sub[:, Symbol("y$k")]) / total for k in 1:K)...,
           ncell = nrow(sub))
    end
    sort!(g, :n)
    return g
end

In [ ]:
n_grid = 1:50
category_names = Dict(
    :duration_multi => ["<5min", "5–15min", "15min–1hr", "1–4hr", "4+hr"],
    :phys_contact   => ["physical", "non-physical"])

panel_outcome = Dict(:dur_home => :duration_multi, :dur_nonhome => :duration_multi,
                     :phys_home => :phys_contact,  :phys_nonhome => :phys_contact)

pp_plots = Dict{Symbol, Plots.Plot}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    chosen = df_waic[df_waic.panel .== panel, :prefer][1]
    chn  = chosen == "constprec" ? chains_constprec[panel] : chains_full[panel]
    mode = chosen == "constprec" ? :constprec : :full

    pp = pp_mu_curves(chn, K, n_grid, mode)
    emp = empirical_proportions_by_n(inp.Y, inp.n, K)
    names = category_names[panel_outcome[panel]]

    plts = []
    for k in 1:K
        p = plot(log.(collect(n_grid)), pp.μ_mean[:, k];
            ribbon = (pp.μ_mean[:, k] .- pp.μ_lower[:, k],
                      pp.μ_upper[:, k] .- pp.μ_mean[:, k]),
            xlabel = "log(degree)", ylabel = "proportion",
            title  = string(names[k]), label = "posterior")
        scatter!(p, log.(emp.n), emp[!, Symbol("p$k")];
            ms = sqrt.(emp.ncell), label = "empirical")
        push!(plts, p)
    end
    pp_plots[panel] = plot(plts...; layout = (1, K), size = (320 * K, 280),
        plot_title = string(panel, " (", chosen, ")"))
    display(pp_plots[panel])
end

In [ ]:
# Sensitivity: drop ALL rows where either outcome is missing.
panels_sens = Dict{Symbol, NamedTuple}()
chains_full_sens      = Dict{Symbol, Chains}()
chains_constprec_sens = Dict{Symbol, Chains}()
waic_rows_sens = NamedTuple[]
for (panel, setting, outcome, K) in [
        (:dur_home,     "home",     :duration_multi, 5),
        (:dur_nonhome,  "non-home", :duration_multi, 5),
        (:phys_home,    "home",     :phys_contact,   2),
        (:phys_nonhome, "non-home", :phys_contact,   2),
    ]
    inp = prepare_dm_inputs(df; setting = setting, outcome = outcome, K = K,
        drop_all_missing = true)
    panels_sens[panel] = inp
    chains_full_sens[panel]      = fit_model_with_forward_mode(
        model_dm_logdeg(inp.X, inp.Y, inp.n; K = K), n_sample; progress = false)
    chains_constprec_sens[panel] = fit_model_with_forward_mode(
        model_dm_logdeg_constprec(inp.X, inp.Y, inp.n; K = K), n_sample; progress = false)
    ll_full      = dm_log_lik_matrix(chains_full_sens[panel],      inp.X, inp.Y, inp.n; K = K, mode = :full)
    ll_constprec = dm_log_lik_matrix(chains_constprec_sens[panel], inp.X, inp.Y, inp.n; K = K, mode = :constprec)
    w_full      = calc_waic_from_loglik(ll_full)
    w_constprec = calc_waic_from_loglik(ll_constprec)
    Δwaic   = w_constprec.waic - w_full.waic
    se_diff = calc_waic_diff_se(w_full.elpd_i, w_constprec.elpd_i)
    push!(waic_rows_sens, (panel = panel, N = size(inp.X, 1),
        waic_full = w_full.waic, waic_constprec = w_constprec.waic,
        Δwaic = Δwaic, se_diff = se_diff,
        prefer = Δwaic > 2 * se_diff ? "full" :
                 Δwaic < -2 * se_diff ? "constprec" : "tie"))
end
df_waic_sens = DataFrame(waic_rows_sens)
df_waic_sens

In [ ]:
# Save chains and WAIC summaries.
out_dir = "../res/2j_proportion_duration_physical"
isdir(out_dir) || mkpath(out_dir)

for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    JLD2.@save joinpath(out_dir, "chn_$(panel)_full.jld2")      chn = chains_full[panel]
    JLD2.@save joinpath(out_dir, "chn_$(panel)_constprec.jld2") chn = chains_constprec[panel]
    JLD2.@save joinpath(out_dir, "chn_$(panel)_full_sens.jld2")      chn = chains_full_sens[panel]
    JLD2.@save joinpath(out_dir, "chn_$(panel)_constprec_sens.jld2") chn = chains_constprec_sens[panel]
end

CSV.write(joinpath(out_dir, "waic_primary.csv"),     df_waic)
CSV.write(joinpath(out_dir, "waic_sensitivity.csv"), df_waic_sens)
println("Saved to ", out_dir)